# End-to-End Data Load Performance: Kerchunk vs netCDF


## Overview

This notebook benchmarks end-to-end variable access performance (open + full `.load()`) for kerchunk and native netCDF datasets. Dataset open time and full variable materialization time are recorded separately to decompose total cost.

These benchmarks represent a workload that fully loads the variable into memory, emphasizing raw I/O throughput in addition to metadata access. Results should be interpreted separately from metadata-only benchmarks or lazy/dask-based compute workflows.

The analysis includes representative cases to isolate structural effects:

- **Amon:** dataset with 1 file vs multi-file (~80–150 files) to contrast fragmentation
- **3hr (normal case):** large time dimension with typical scaling behavior
- **3hr (time-last outlier):** extremely large time dimension ordered last, where kerchunk exhibits higher load cost

The goal is to evaluate how file count, temporal frequency, and dimension structure influence total access time and relative scaling between kerchunk and native netCDF.


In [1]:
import json
import time

import numpy as np
import pandas as pd
import xcdat as xc
from IPython.display import HTML

## Read in Results and Input Mapping


In [2]:
df_raw = pd.read_csv(
    "riotai/results/20260126_130127/kerchunk_vs_netcdf_raw_20260126_130127.csv"
)

# Create a DataFrame from the JSON to NetCDF mapping.
with open("riotai/json_to_netcdf_maps/json_to_netcdf.json", "r") as file:
    json_netcdf_map = json.load(file)

rows = []
for freq, json_map in json_netcdf_map.items():
    for json_key, netcdf_filepaths in json_map.items():
        rows.append(
            {
                "frequency": freq,
                "json": json_key,
                "netcdf_filepaths": netcdf_filepaths,
            }
        )

df_json_netcdf = pd.DataFrame(rows)

df_raw_joined = df_raw.merge(df_json_netcdf, how="left", on=["frequency", "json"])

## Get specific datasets for these cases:

- **Amon:** dataset with 1 file vs ~100 files (fragmentation comparison)
- **3hr (normal case):** large time dimension with typical performance
- **3hr (time-last outlier):** very large time dimension ordered last, where kerchunk shows higher setup cost


In [3]:
# Filter for Amon: 1 file vs ~100 files
df_amon_1_file = df_raw_joined[
    (df_raw_joined["frequency"] == "Amon") & (df_raw_joined["num_netcdf_files"] == 1)
]
df_amon_many_files = df_raw_joined[
    (df_raw_joined["frequency"] == "Amon") & (df_raw_joined["num_netcdf_files"] >= 80)
]

# Filter for daily (normal case): large time dimension, typical performance
df_daily_normal = df_raw_joined[
    (df_raw_joined["frequency"] == "day")
    & (df_raw_joined["timesteps"] > 100000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

# Filter for 3hr (normal case): large time dimension, typical performance
df_3hr_normal = df_raw_joined[
    (df_raw_joined["frequency"] == "3hr")
    & (df_raw_joined["timesteps"] > 100000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

# Filter for 3hr (time-last outlier): very large time dimension ordered last
df_3hr_time_last_outlier = df_raw_joined[
    (df_raw_joined["frequency"] == "3hr")
    & (df_raw_joined["timesteps"] > 500000)
    & (df_raw_joined["num_netcdf_files"] < 20)
]

df_amon_1_file = df_amon_1_file.head(1).iloc[0]
df_amon_many_files = df_amon_many_files.head(1).iloc[0]
df_3hr_normal = df_3hr_normal.head(1).iloc[0]
df_3hr_time_last_outlier = df_3hr_time_last_outlier.head(1).iloc[0]

In [ ]:
import time
import numpy as np
import pandas as pd
import xarray as xr


def benchmark_variable_load(
    kerchunk_path,
    netcdf_paths,
    variable,
    runs: int = 5,
    decode_cf: bool = True,
    mask_and_scale: bool = True,
    drop_first_run: bool = False,
) -> pd.DataFrame:
    """
    Benchmark xarray open time and variable materialization time (`.load()`)
    for kerchunk and native netCDF backends.

    Both backends are opened with `chunks={}` to produce Dask-backed lazy arrays.
    The benchmark measures:

      1. Dataset open time (metadata parsing + Dask graph construction)
      2. Variable load time via `.load()` (Dask execution, IO, decoding,
         masking/scaling, and in-memory materialization)

    Notes
    -----
    - This benchmark reflects full xarray user-facing behavior, not raw IO.
    - `.load()` includes scheduler overhead and decoding costs.
    - Repeated runs reflect warm filesystem cache behavior.
    - If `drop_first_run=True`, the first iteration (often cold cache)
      is excluded from median statistics.
    - Kerchunk and netCDF chunk structures may differ unless explicitly
      controlled outside this function.

    Parameters
    ----------
    kerchunk_path : str
        Path to kerchunk reference JSON.
    netcdf_paths : str or list[str]
        Path or list of netCDF files.
    variable : str
        Variable name to materialize (e.g., "tas", "pr").
    runs : int, default=5
        Number of repetitions.
    decode_cf : bool, default=True
        Whether to apply CF decoding.
    mask_and_scale : bool, default=True
        Whether to apply masking and scale/offset decoding.
    drop_first_run : bool, default=False
        Exclude first run from median statistics.

    Returns
    -------
    pd.DataFrame
        Per-run open/load timings and corresponding medians.
    """
    kc_load_times = []
    nc_load_times = []
    kc_open_times = []
    nc_open_times = []

    for _ in range(runs):
        # ---- Kerchunk ----
        t0 = time.perf_counter()
        ds = xr.open_dataset(
            kerchunk_path,
            engine="kerchunk",
            chunks={},
            decode_cf=decode_cf,
            mask_and_scale=mask_and_scale,
        )
        t1 = time.perf_counter()
        kc_open_times.append(t1 - t0)

        var = ds[variable]
        t0 = time.perf_counter()
        var.load()
        t1 = time.perf_counter()
        kc_load_times.append(t1 - t0)
        ds.close()

        # ---- Native netCDF ----
        if isinstance(netcdf_paths, (list, tuple)) and len(netcdf_paths) > 1:
            t0 = time.perf_counter()
            ds = xr.open_mfdataset(
                netcdf_paths,
                combine="by_coords",
                parallel=False,
                chunks={},
                decode_cf=decode_cf,
                mask_and_scale=mask_and_scale,
            )
            t1 = time.perf_counter()
            nc_open_times.append(t1 - t0)
        else:
            path = (
                netcdf_paths[0]
                if isinstance(netcdf_paths, (list, tuple))
                else netcdf_paths
            )
            t0 = time.perf_counter()
            ds = xr.open_dataset(
                path,
                chunks={},
                decode_cf=decode_cf,
                mask_and_scale=mask_and_scale,
            )
            t1 = time.perf_counter()
            nc_open_times.append(t1 - t0)

        var = ds[variable]
        t0 = time.perf_counter()
        var.load()
        t1 = time.perf_counter()
        nc_load_times.append(t1 - t0)
        ds.close()

    # Optional warm-cache median
    kc_load_eval = kc_load_times[1:] if drop_first_run else kc_load_times
    nc_load_eval = nc_load_times[1:] if drop_first_run else nc_load_times
    kc_open_eval = kc_open_times[1:] if drop_first_run else kc_open_times
    nc_open_eval = nc_open_times[1:] if drop_first_run else nc_open_times

    kc_load_med = float(np.median(kc_load_eval))
    nc_load_med = float(np.median(nc_load_eval))
    kc_open_med = float(np.median(kc_open_eval))
    nc_open_med = float(np.median(nc_open_eval))

    data = {
        "kerchunk_open_runs": kc_open_times,
        "netcdf_open_runs": nc_open_times,
        "kerchunk_load_runs": kc_load_times,
        "netcdf_load_runs": nc_load_times,
        "kerchunk_open_median": [kc_open_med] * runs,
        "netcdf_open_median": [nc_open_med] * runs,
        "kerchunk_load_median": [kc_load_med] * runs,
        "netcdf_load_median": [nc_load_med] * runs,
        "difference_load_median": [kc_load_med - nc_load_med] * runs,
    }

    return pd.DataFrame(data)

## Case 1 - Amon (1 file dataset)


In [ ]:
def benchmark_variable_load(
    kerchunk_path,
    netcdf_paths,
    variable,
    runs=5,
) -> pd.DataFrame:
    """
    Benchmark dataset open time and variable materialization time (`.load()`)
    for kerchunk and native netCDF datasets using xcdat.

    Both backends are opened with `chunks={}` to force dask-backed lazy arrays.
    This ensures `.load()` triggers comparable compute on both sides. Without
    this, single-file opens may return eager arrays while multi-file opens use
    dask, leading to inconsistent load-time measurements.

    Open and load times are recorded separately. Load timings reflect warm
    filesystem cache behavior after the first iteration.

    Parameters
    ----------
    kerchunk_path : str
        Path to kerchunk JSON file.
    netcdf_paths : str or list
        Path or list of netCDF files.
    variable : str
        Variable name to load (e.g., "tas", "pr").
    runs : int
        Number of repetitions.

    Returns
    -------
    pd.DataFrame
        DataFrame with per-run open/load times and median statistics for both backends.
    """
    kc_load_times = []
    nc_load_times = []
    kc_open_times = []
    nc_open_times = []

    for _ in range(runs):
        # ---- Kerchunk (lazy / dask-backed) ----
        t0 = time.perf_counter()
        ds = xc.open_dataset(kerchunk_path, engine="kerchunk", chunks={})
        t1 = time.perf_counter()
        kc_open_times.append(t1 - t0)

        t0 = time.perf_counter()
        ds[variable].load()
        t1 = time.perf_counter()
        ds.close()
        kc_load_times.append(t1 - t0)

        # ---- Native netCDF (match execution mode) ----
        if isinstance(netcdf_paths, (list, tuple)) and len(netcdf_paths) > 1:
            t0 = time.perf_counter()
            ds = xc.open_mfdataset(
                netcdf_paths, combine="by_coords", parallel=False, chunks={}
            )
            t1 = time.perf_counter()
            nc_open_times.append(t1 - t0)
        else:
            path = (
                netcdf_paths[0]
                if isinstance(netcdf_paths, (list, tuple))
                else netcdf_paths
            )
            t0 = time.perf_counter()
            ds = xc.open_dataset(path, chunks={})
            t1 = time.perf_counter()
            nc_open_times.append(t1 - t0)

        t0 = time.perf_counter()
        ds[variable].load()
        t1 = time.perf_counter()
        ds.close()
        nc_load_times.append(t1 - t0)

    # Store per-run values as lists, not repeated per row
    data = {
        "kerchunk_load_runs": kc_load_times,
        "netcdf_load_runs": nc_load_times,
        "kerchunk_open_runs": kc_open_times,
        "netcdf_open_runs": nc_open_times,
        "kerchunk_load_median": float(np.median(kc_load_times)),
        "netcdf_load_median": float(np.median(nc_load_times)),
        "kerchunk_open_median": float(np.median(kc_open_times)),
        "netcdf_open_median": float(np.median(nc_open_times)),
        "difference_load_median": float(
            np.median(kc_load_times) - np.median(nc_load_times)
        ),
    }
    return pd.DataFrame(data)

In [142]:
df_amon_1_file

frequency                                                        Amon
json                /global/cfs/projectdirs/m4931/kerchu...
num_netcdf_files                                                    1
timesteps                                                        1980
dims                {'time': 1980, 'lat': 128, 'lon': 256, 'axis_n...
kerchunk_time                                                 0.51895
netcdf_time                                                  0.165315
netcdf_filepaths    [/global/cfs/projectdirs/m4931/gsharing/css03_...
Name: 0, dtype: object

In [ ]:
results_amon_1_file = benchmark_variable_load(
    kerchunk_path=df_amon_1_file["json"],
    netcdf_paths=df_amon_1_file["netcdf_filepaths"],
    variable="tas",
    runs=5,
)

In [130]:
results_amon_1_file

,kerchunk_load_runs,netcdf_load_runs,kerchunk_open_runs,netcdf_open_runs,kerchunk_load_median,netcdf_load_median,kerchunk_open_median,netcdf_open_median,difference_load_median
0,"[18.98409984598402, 18.195360581041314, 16.577...","[2.5239939930615947, 2.3860836980165914, 2.459...","[0.21254660398699343, 0.08586538792587817, 0.0...","[0.09447447198908776, 0.10652644198853523, 0.1...",18.198937,2.459899,0.085865,0.105991,15.739038


**Key observations:**

- This Amon dataset contains 1,980 timesteps on a 128 × 256 grid and is stored in a single netCDF file.
- Kerchunk and native netCDF have comparable dataset open times (~0.09–0.11 s).
- Full variable materialization (`.load()`) is substantially slower for kerchunk (~18 s) than for native netCDF (~2.5 s).
- The performance difference arises during data loading rather than metadata access.

**Takeaway:**
For a single contiguous netCDF file of moderate size, native netCDF provides significantly faster full-variable loads. In the absence of file fragmentation, kerchunk introduces additional indirection overhead and does not provide a performance advantage.


## Case 2 - Amon (~100 file dataset)


In [139]:
df_amon_many_files

frequency                                                        Amon
json                /global/cfs/projectdirs/m4931/kerchu...
num_netcdf_files                                                   86
timesteps                                                        1032
dims                {'lat': 256, 'bnds': 2, 'lon': 512, 'time': 1032}
kerchunk_time                                                0.668246
netcdf_time                                                  7.595436
netcdf_filepaths    [/global/cfs/projectdirs/m4931/gsharing/css03_...
Name: 3, dtype: object

In [ ]:
results_amon_many_files = benchmark_variable_load(
    kerchunk_path=df_amon_many_files["json"],
    netcdf_paths=df_amon_many_files["netcdf_filepaths"],
    variable="tas",
)

In [143]:
results_amon_many_files

,kerchunk_load_runs,netcdf_load_runs,kerchunk_open_runs,netcdf_open_runs,kerchunk_load_median,netcdf_load_median,kerchunk_open_median,netcdf_open_median,difference_load_median
0,"[7.877022645901889, 8.818493606057018, 9.42541...","[4.582968450966291, 4.726152278017253, 4.91925...","[0.38614483503624797, 0.07125870196614414, 0.0...","[5.042024420923553, 4.407715269015171, 4.66535...",8.903087,4.698441,0.073798,4.680304,4.204646


**Key observations:**

- This Amon dataset contains 1,032 timesteps on a 256 × 512 grid and is split across 86 netCDF files.
- Kerchunk open time (~0.07 s) is dramatically faster than native netCDF open time (~4.7 s), reflecting the cost of multi-file discovery and coordination in netCDF.
- During full variable materialization (`.load()`), kerchunk (~8.9 s) is slower than native netCDF (~4.7 s).
- The primary performance difference between backends appears during dataset open rather than data load.

**Takeaway:**
For fragmented multi-file Amon datasets, kerchunk substantially reduces dataset open time by avoiding file-by-file coordination overhead. However, during full data materialization, native netCDF remains faster for this case. Kerchunk’s advantage in this regime is driven primarily by metadata and file aggregation efficiency rather than raw data read speed.


## Case 3 - Day


In [ ]:
df_amon_many_files
results_amon_many_files = benchmark_variable_load(
    kerchunk_path=df_amon_many_files["json"],
    netcdf_paths=df_amon_many_files["netcdf_filepaths"],
    variable="tas",
)
results_amon_many_files

## Case 3 - 3hr (normal-case)


In [144]:
df_3hr_normal

frequency                                                         3hr
json                /global/cfs/projectdirs/m4931/kerchu...
num_netcdf_files                                                    7
timesteps                                                      189928
dims                {'time': 189928, 'lat': 128, 'lon': 256, 'axis...
kerchunk_time                                                4.073678
netcdf_time                                                 28.071928
netcdf_filepaths    [/global/cfs/projectdirs/m4931/gsharing/css03_...
Name: 89, dtype: object

In [148]:
results_3hr_normal = benchmark_variable_load(
    kerchunk_path=df_3hr_normal["json"],
    netcdf_paths=df_3hr_normal["netcdf_filepaths"],
    variable="pr",
)

KeyboardInterrupt: 

In [ ]:
results_3hr_normal

NameError: name 'results_3hr_normal_case' is not defined

## TL;DR

### Dataset Open (`.open_dataset()`)

- **Single-file datasets (e.g., Amon, 1 file):**  
  Kerchunk and native netCDF have similar open times. NetCDF may be slightly faster since it reads metadata directly from a single file.

- **Multi-file, fragmented datasets (e.g., 50–100+ files):**  
  Kerchunk is significantly faster to open. It avoids file-by-file discovery and aggregation overhead that native netCDF incurs with `open_mfdataset`.

- **High-frequency datasets (e.g., 3hr, hourly):**  
  As file count and fragmentation increase, kerchunk’s advantage in open time becomes more pronounced.

---

### Variable Load (`.load()`)

- **Single-file, moderate-size datasets (~1–2k timesteps):**  
  Native netCDF is typically faster for full `.load()`. Kerchunk adds indirection overhead and does not improve raw contiguous read performance.

- **Multi-file datasets:**  
  Native netCDF may still be faster for raw data materialization, but kerchunk can offset this with much faster open times, making end-to-end performance competitive.

- **Very large time dimensions (e.g., 3hr with 100k+ timesteps):**  
  Load performance becomes sensitive to chunking and dimension order. Large, finely chunked time axes—especially when `time` is last—can increase kerchunk load cost.

---

### Chunking and Execution Mode

- Using `chunks={}` forces dask-backed lazy arrays for both backends and ensures fair `.load()` comparisons.
- Native netCDF benefits from optimized C-level sequential reads for single contiguous files.
- Kerchunk’s strength lies in metadata aggregation and fragmentation handling, not raw single-file throughput.

---

**Bottom line:**  
Kerchunk excels at reducing dataset open cost for fragmented, multi-file datasets. Native netCDF remains most efficient for full-variable loads

| Scenario               | Winner              |
| ---------------------- | ------------------- |
| 1 file + full load     | NetCDF              |
| Many files + open      | Kerchunk            |
| End-to-end, fragmented | Often competitive   |
| Huge time axis         | Depends on chunking |
